# Train and Evaluate — EfficientNet Face2Face Holdout

This notebook is configured for Obinna's assigned run:

`efficientnet_holdout-face2face`

It is adapted for the Northeastern Explorer **Tesla P100 12 GB** session:

- uses the project `.venv` Python for every script;
- uses `fp16` instead of `bf16` because the P100 does not support native BF16;
- lowers the local batch size to 16 to reduce the chance of GPU out-of-memory errors;
- does not modify the committed YAML config;
- validates the crop manifest, split file, dependencies, GPU, checkpoint, and result JSON.

Prerequisite: the preprocessing notebook must have created
`data/manifests/crops.parquet` and the crop cache under `data/processed/`.


## 0. Select and verify the assigned run

This notebook deliberately uses your claimed run rather than automatically selecting
the first unfinished run from the board.

In [1]:
from pathlib import Path
import os
import sys
import glob
import subprocess

REPO_ROOT = Path.home() / "Cross-Generator-Generalization-in-Deepfake-Detection"

if not REPO_ROOT.exists():
    raise FileNotFoundError(f"Repository not found: {REPO_ROOT}")

os.chdir(REPO_ROOT)

print("Repository:", Path.cwd())
print("Python:", sys.executable)

if ".venv/bin/python" not in sys.executable:
    raise RuntimeError(
        "Wrong notebook kernel. Select 'Python (.venv - Deepfake)' before continuing."
    )

RUN = "efficientnet_holdout-face2face"

runs = sorted(
    Path(f).stem
    for f in glob.glob("configs/*.yaml")
    if "preprocess" not in Path(f).name
    and ".smoke" not in Path(f).name
)

if RUN not in runs:
    raise FileNotFoundError(
        f"Config for {RUN} was not found. Available runs: {runs}"
    )

print("\nAssigned run:", RUN)

# P100-compatible machine-local overrides.
AMP_OVERRIDE = "fp16"
BATCH_OVERRIDE = 16

print("Local AMP override:", AMP_OVERRIDE)
print("Local batch-size override:", BATCH_OVERRIDE)

Repository: /home/okonkwo.ob/Cross-Generator-Generalization-in-Deepfake-Detection
Python: /home/okonkwo.ob/Cross-Generator-Generalization-in-Deepfake-Detection/.venv/bin/python

Assigned run: efficientnet_holdout-face2face
Local AMP override: fp16
Local batch-size override: 16


## 1. Validate dependencies and GPU

`timm` must be installed in the same `.venv` used by this notebook. The GPU should
appear as a Tesla P100.

In [2]:
import importlib.util
import torch

missing = [
    package
    for package in ["timm", "yaml", "pandas", "sklearn"]
    if importlib.util.find_spec(package) is None
]

if missing:
    raise ModuleNotFoundError(
        "Missing packages in the project .venv: "
        + ", ".join(missing)
        + ". Install them from the terminal with: "
        + f"{sys.executable} -m pip install timm pyyaml pandas scikit-learn"
    )

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Use a GPU Explorer session.")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA capability:", torch.cuda.get_device_capability(0))

PyTorch: 2.6.0+cu124
CUDA available: True
GPU: Tesla V100-SXM2-32GB
CUDA capability: (7, 0)


## 1. Confirm the crops cache is present
If this fails, run `00_setup_and_preprocess.ipynb` first.

In [3]:
from pathlib import Path
import pandas as pd

manifest_path = Path("data/manifests/crops.parquet")

if not manifest_path.exists():
    raise FileNotFoundError(
        "data/manifests/crops.parquet is missing. Complete preprocessing first."
    )

m = pd.read_parquet(manifest_path)

if m.empty:
    raise RuntimeError("The crop manifest exists but contains no rows.")

print("Total crops:", len(m))
print("\nCrops by method:")
print(m.groupby("method").size())

required_methods = {"real", "DeepFakes", "Face2Face", "FaceSwap", "NeuralTextures"}
present_methods = set(m["method"].astype(str).unique())
missing_methods = required_methods - present_methods

if missing_methods:
    raise RuntimeError(
        "The crop manifest is incomplete. Missing methods: "
        + ", ".join(sorted(missing_methods))
    )

Total crops: 97262

Crops by method:
method
DeepFakes         19433
Face2Face         19455
FaceSwap          19457
NeuralTextures    19465
real              19452
dtype: int64


## 2. Build the leave-one-manipulation-out splits
Writes one identity-disjoint fold per held-out method into `data/splits/`.
For each fold, train+val are 3 methods plus real, test is the held-out method plus real.

In [4]:
from pathlib import Path
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "data/make_splits.py",
        "--manifest", "data/manifests/crops.parquet",
        "--out", "data/splits",
    ],
    check=True,
)

split_files = sorted(Path("data/splits").glob("*"))
print(f"Created/found {len(split_files)} split files:")
for path in split_files:
    print(" ", path)

holdout-deepfakes.csv 97262 rows
holdout-face2face.csv 97262 rows
holdout-faceswap.csv 97262 rows
holdout-neuraltextures.csv 97262 rows
Created/found 4 split files:
  data/splits/holdout-deepfakes.csv
  data/splits/holdout-face2face.csv
  data/splits/holdout-faceswap.csv
  data/splits/holdout-neuraltextures.csv


## 3. Load your run config
Loads the committed config for the `RUN` you picked in step 0 and validates its inputs.

The optional overrides from step 0 (`AMP_OVERRIDE` / `BATCH_OVERRIDE`) adapt to your GPU
without editing the committed config: when set, the notebook writes a gitignored
`configs/_local/` copy and trains from that, so the committed config stays authoritative
(seed, data, split, lr are never touched -- only how the run executes).

In [5]:
from pathlib import Path
import yaml

cfg_file = Path("configs") / f"{RUN}.yaml"

with cfg_file.open("r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)

cfg["amp"] = AMP_OVERRIDE
cfg["batch_size"] = BATCH_OVERRIDE
run_name = cfg["run_name"]

local_config_dir = Path("configs/_local")
local_config_dir.mkdir(parents=True, exist_ok=True)
cfg_path = local_config_dir / f"{run_name}.yaml"

with cfg_path.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(cfg, handle, sort_keys=False)

print("Committed config:", cfg_file)
print("Machine-local config:", cfg_path)

for key in ["manifest", "split"]:
    required_path = Path(cfg[key])
    if not required_path.exists():
        raise FileNotFoundError(f"Required config path is missing: {required_path}")

print("\nEffective training config:")
print(yaml.safe_dump(cfg, sort_keys=False))

Committed config: configs/efficientnet_holdout-face2face.yaml
Machine-local config: configs/_local/efficientnet_holdout-face2face.yaml

Effective training config:
run_name: efficientnet_holdout-face2face
seed: 1337
backbone: efficientnet_b4
pretrained: true
input_size: 224
manifest: data/manifests/crops.parquet
split: data/splits/holdout-face2face.csv
held_out_method: Face2Face
epochs: 15
batch_size: 16
lr: 0.0003
amp: fp16
checkpoint_dir: checkpoints/efficientnet_holdout-face2face
video_level: true
extra_test_sets:
- name: SimSwap
  split: data/splits/simswap-test.csv



## 4. Train
Fine-tunes the ImageNet-pretrained backbone on the 3 training methods plus real.
Saves a checkpoint under `checkpoints/<run_name>/`. Uses the GPU if one is visible
(check step 3 of the setup notebook). This is the long cell.

In [6]:
from pathlib import Path
import subprocess
import sys

print("Starting training:", run_name)
print("Config:", cfg_path)

subprocess.run(
    [
        sys.executable,
        "experiments/train.py",
        "--config", str(cfg_path),
    ],
    check=True,
)

checkpoint_dir = Path(cfg.get("checkpoint_dir", f"checkpoints/{run_name}"))

if not checkpoint_dir.exists():
    raise FileNotFoundError(
        f"Training finished but checkpoint directory was not found: {checkpoint_dir}"
    )

print("Training completed.")
print("Checkpoint directory:", checkpoint_dir)

Starting training: efficientnet_holdout-face2face
Config: configs/_local/efficientnet_holdout-face2face.yaml


epoch 1/15  train_loss 0.2235
           val_acc 0.0000
epoch 2/15  train_loss 0.0758
           val_acc 0.0000
epoch 3/15  train_loss 0.0489
           val_acc 0.0000
epoch 4/15  train_loss 0.0364
           val_acc 0.0000
epoch 5/15  train_loss 0.0312
           val_acc 0.0000
epoch 6/15  train_loss 0.0269
           val_acc 0.0000
epoch 7/15  train_loss 0.0259
           val_acc 0.0000
epoch 8/15  train_loss 0.0238
           val_acc 0.0000
epoch 9/15  train_loss 0.0237
           val_acc 0.0000
epoch 10/15  train_loss 0.0213
           val_acc 0.0000
epoch 11/15  train_loss 0.0222
           val_acc 0.0000
epoch 12/15  train_loss 0.0205
           val_acc 0.0000
epoch 13/15  train_loss 0.0207
           val_acc 0.0000
epoch 14/15  train_loss 0.0202
           val_acc 0.0000
epoch 15/15  train_loss 0.0217
           val_acc 0.0000
saved checkpoints/efficientnet_holdout-face2face/model.pt
Training completed.
Checkpoint directory: checkpoints/efficientnet_holdout-face2face


## 5. Evaluate
Scores the trained model on each method: the 3 seen methods (in-distribution) and
the held-out method (unseen), plus the SimSwap set if its split exists. Writes
`experiments/results/<run_name>.json` in the shared results schema.

In [7]:
from pathlib import Path
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "experiments/evaluate.py",
        "--config", str(cfg_path),
    ],
    check=True,
)

result_path = Path("experiments/results") / f"{run_name}.json"

if not result_path.exists():
    raise FileNotFoundError(
        f"Evaluation completed but result JSON was not found: {result_path}"
    )

print("Evaluation completed.")
print("Result:", result_path)

/home/okonkwo.ob/Cross-Generator-Generalization-in-Deepfake-Detection/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


skip SimSwap (no split file yet)
wrote experiments/results/efficientnet_holdout-face2face.json
Face2Face        seen=False auc=nan acc=0.4346
Evaluation completed.
Result: experiments/results/efficientnet_holdout-face2face.json


## 6. Read the result (the seen-vs-unseen gap)
The headline number is the drop from seen methods to the held-out (unseen) one.

In [8]:
from pathlib import Path
import json
import pandas as pd

result_path = Path("experiments/results") / f"{run_name}.json"

with result_path.open("r", encoding="utf-8") as handle:
    r = json.load(handle)

df = pd.DataFrame(r["results"])

print(
    f"run: {r['run_name']}  "
    f"backbone: {r['backbone']}  "
    f"held-out: {r['held_out_method']}  "
    f"level: {r['level']}"
)

print(df[["tested_on", "seen", "auc", "acc", "f1"]].to_string(index=False))

seen_auc = df.loc[df["seen"] == True, "auc"].mean()
unseen_auc = df.loc[df["seen"] == False, "auc"].mean()

print(f"\nMean seen AUC:   {seen_auc:.4f}")
print(f"Mean unseen AUC: {unseen_auc:.4f}")
print(f"Generalization gap: {seen_auc - unseen_auc:.4f}")

run: efficientnet_holdout-face2face  backbone: efficientnet_b4  held-out: Face2Face  level: video
tested_on  seen  auc    acc     f1
Face2Face False  NaN 0.4346 0.6059

Mean seen AUC:   nan
Mean unseen AUC: nan
Generalization gap: nan


## 7. Share your result with the team
Two small pushes: your results JSON (feeds the transfer matrix) and your row in
`RUNS.md` marked done. Checkpoints and crops are gitignored, so they never get pushed.
Without step 7 your run stays on your VM and the board still shows it TODO.

First time on a VM: make sure git is configured (`git config user.name` / `user.email`)
and that you can push (you are a repo collaborator; cloning over HTTPS caches your token).

In [9]:
run_json = f"experiments/results/{run_name}.json"
assert os.path.exists(run_json), "no results JSON yet; run steps 4 and 5 first"
print("remember to mark", run_name, "done in RUNS.md, then:")
print("  git add", run_json, "RUNS.md")
print(f"  git commit -m 'results: {run_name}'")
print("  git pull --rebase && git push")
# uncomment to do it from here:
# !git add {run_json} RUNS.md
# !git commit -m "results: {run_name}"
# !git pull --rebase
# !git push

remember to mark efficientnet_holdout-face2face done in RUNS.md, then:
  git add experiments/results/efficientnet_holdout-face2face.json RUNS.md
  git commit -m 'results: efficientnet_holdout-face2face'
  git pull --rebase && git push


## Next steps
- Check the board in step 0: when all eight show DONE, assemble them with
  `02_transfer_matrix.ipynb`.
- Taking a second run? Change `RUN` in step 0 (claim it in `RUNS.md` first) and re-run.
- The held-out method is never opened during training, so its column is a clean
  unseen-generator measurement.
- The `SimSwap` column stays empty until the self-generated set and its split exist.